# 问题3：大规模单车辆调度算法设计与实现

## 基于迭代聚类窗口方法满足550量子比特硬件限制

### 算法流程
1. 时空聚类：将50个客户点按时空特征聚类为4-5个簇
2. 滑动窗口优化：在每个簇内使用10-15点的滑动窗口进行局部QUBO优化
3. 迭代改进：评估边界点并优化聚类分配
4. 路径整合：将局部优化路径整合为全局路径

### 硬件限制
- 最大QUBO问题规模：550量子比特
- 每个滑动窗口：10-15个节点（对应100-225量子比特）
- 总节点数：50个客户点 + 1个仓库点

In [ ]:
# 导入基础库
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# 导入MDS用于降维可视化
from sklearn.manifold import MDS

# 导入自定义模块
import sys
sys.path.append('.')

from spatiotemporal_clustering import (
    compute_spatiotemporal_distance_matrix,
    improved_kmeans_clustering,
    spatiotemporal_clustering
)

from sliding_window_optimizer import (
    solve_window_qubo,
    sliding_window_optimization
)

from iterative_refinement import (
    evaluate_boundary_violations,
    migrate_boundary_points,
    iterative_refinement_controller
)

from visualization_utils import (
    plot_clustering_results,
    plot_window_optimization,
    plot_iteration_progress,
    plot_hierarchical_path
)

# 设置可视化主题
sns.set_theme(style="whitegrid", palette="husl")
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 设置随机种子
np.random.seed(42)

print("所有库和模块导入完成！")

## 1. 数据加载与预处理

In [ ]:
# 加载50节点完整数据
print("正在加载数据...")
data_path = "v1/参考算例.xlsx"

# 读取节点属性表
node_data = pd.read_excel(data_path, sheet_name="节点属性信息")
print(f"节点属性表形状: {node_data.shape}")
print("节点属性表前5行:")
print(node_data.head())

# 提取节点ID、时间窗、服务时间
node_ids = node_data['节点ID'].values
a_i = node_data['开始服务时间下界'].values  # 时间窗下界
b_i = node_data['开始服务时间上界'].values  # 时间窗上界
s_i = node_data['服务时间'].values  # 服务时间

print(f"\n节点数量: {len(node_ids)}")
print(f"时间窗范围: [{a_i.min():.1f}, {b_i.max():.1f}]")
print(f"平均服务时间: {s_i.mean():.2f}")

# 读取旅行时间矩阵
travel_time_data = pd.read_excel(data_path, sheet_name="旅行时间矩阵")
print(f"\n旅行时间矩阵形状: {travel_time_data.shape}")

# 提取旅行时间矩阵（51×51，包含仓库）
travel_time_matrix = travel_time_data.iloc[:, 1:].values  # 跳过第一列（节点编号）
print(f"旅行时间矩阵形状（处理后）: {travel_time_matrix.shape}")

# 基本统计信息
print("\n旅行时间矩阵统计信息:")
print(f"最小值: {travel_time_matrix.min():.2f}")
print(f"最大值: {travel_time_matrix.max():.2f}")
print(f"平均值: {travel_time_matrix.mean():.2f}")
print(f"标准差: {travel_time_matrix.std():.2f}")

# 检查对称性（应为对称矩阵）
is_symmetric = np.allclose(travel_time_matrix, travel_time_matrix.T)
print(f"矩阵是否对称: {is_symmetric}")

# 创建完整的数据字典
data = {
    'node_ids': node_ids,
    'a_i': a_i,
    'b_i': b_i,
    's_i': s_i,
    'travel_time_matrix': travel_time_matrix,
    'num_nodes': len(node_ids),
    'depot_id': 0  # 假设仓库节点编号为0
}

print("\n数据加载完成！")

## 2. 时空聚类分析

In [ ]:
# 执行时空联合聚类
print("正在进行时空联合聚类...")

# 调用聚类函数
clustering_results = spatiotemporal_clustering(
    node_ids=data['node_ids'],
    dist_matrix=data['travel_time_matrix'],  # 注意参数名是dist_matrix
    a_i=data['a_i'],
    b_i=data['b_i'],
    n_clusters=5  # 目标聚类数
)

# 提取聚类结果
cluster_labels = clustering_results['cluster_labels']
cluster_centers = clustering_results['cluster_centers']
num_clusters = clustering_results['num_clusters']

print(f"\n聚类完成！共生成 {num_clusters} 个簇")
print("各簇大小:")
for cluster_id in range(num_clusters):
    cluster_size = np.sum(cluster_labels == cluster_id)
    print(f"  簇 {cluster_id}: {cluster_size} 个节点")

# 计算各簇的平均时间窗
print("\n各簇平均时间窗:")
for cluster_id in range(num_clusters):
    cluster_indices = np.where(cluster_labels == cluster_id)[0]
    avg_a = data['a_i'][cluster_indices].mean()
    avg_b = data['b_i'][cluster_indices].mean()
    print(f"  簇 {cluster_id}: [{avg_a:.1f}, {avg_b:.1f}]")

# 计算MDS坐标用于可视化
print("\n计算MDS降维坐标用于可视化...")
distance_matrix = compute_spatiotemporal_distance_matrix(
    dist_matrix=data['travel_time_matrix'],
    a_i=data['a_i'],
    b_i=data['b_i'],
    spatial_weight=0.5,  # 空间权重
    temporal_weight=0.5  # 时间权重
)

# 使用MDS降维到2D
mds = MDS(n_components=2, dissimilarity='precomputed', random_state=42, normalized_stress='auto')
mds_coords = mds.fit_transform(distance_matrix)

print("MDS降维完成！")

# 可视化聚类结果
print("\n生成聚类可视化...")
fig = plot_clustering_results(
    mds_coords=mds_coords,
    cluster_labels=cluster_labels,
    node_ids=data['node_ids'],
    a_i=data['a_i'],
    b_i=data['b_i'],
    cluster_centers=cluster_centers
)

plt.tight_layout()
plt.show()

print("时空聚类分析完成！")

## 3. 簇间路径规划与簇内滑动窗口优化

In [ ]:
# 步骤3: 簇间路径规划
print("正在进行簇间路径规划...")

# 确定簇间访问顺序（使用簇中心作为超节点）
print("计算簇中心作为超节点...")

# 提取簇中心节点（假设每个簇的中心节点是簇内节点）
super_nodes = []
for cluster_id in range(num_clusters):
    cluster_indices = np.where(cluster_labels == cluster_id)[0]
    # 选择时间窗最接近平均值的节点作为簇代表
    avg_a = data['a_i'][cluster_indices].mean()
    avg_b = data['b_i'][cluster_indices].mean()
    
    # 计算每个节点时间窗与平均值的距离
    time_window_distances = np.abs(data['a_i'][cluster_indices] - avg_a) + np.abs(data['b_i'][cluster_indices] - avg_b)
    center_idx = cluster_indices[np.argmin(time_window_distances)]
    super_nodes.append(center_idx)

print(f"超节点（簇代表）: {super_nodes}")

# 计算超节点距离矩阵
super_node_dist_matrix = np.zeros((num_clusters, num_clusters))
for i in range(num_clusters):
    for j in range(num_clusters):
        if i != j:
            super_node_dist_matrix[i, j] = data['travel_time_matrix'][super_nodes[i], super_nodes[j]]

print(f"超节点距离矩阵形状: {super_node_dist_matrix.shape}")

# 使用最近邻启发式确定簇间访问顺序
print("使用最近邻启发式确定簇间访问顺序...")
visited = [False] * num_clusters
cluster_order = [0]  # 从簇0开始（假设包含仓库的簇）
visited[0] = True

current_cluster = 0
for _ in range(num_clusters - 1):
    # 找到未访问簇中距离当前簇最近的簇
    unvisited = [i for i in range(num_clusters) if not visited[i]]
    nearest_cluster = unvisited[np.argmin(super_node_dist_matrix[current_cluster, unvisited])]
    cluster_order.append(nearest_cluster)
    visited[nearest_cluster] = True
    current_cluster = nearest_cluster

print(f"簇间访问顺序: {cluster_order}")

# 计算簇间路径总距离
inter_cluster_distance = 0
for i in range(len(cluster_order) - 1):
    from_cluster = cluster_order[i]
    to_cluster = cluster_order[i + 1]
    inter_cluster_distance += super_node_dist_matrix[from_cluster, to_cluster]

print(f"簇间路径总距离: {inter_cluster_distance:.2f}")

print("簇间路径规划完成！")

# 步骤4: 簇内滑动窗口优化
print("\n" + "="*50)
print("开始簇内滑动窗口优化...")

# 初始化存储变量
intra_cluster_paths = []
total_intra_distance = 0
total_penalty = 0
window_size = 10
step_size = 5

print(f"滑动窗口参数: window_size={window_size}, step_size={step_size}")

# 按簇间顺序处理每个簇
for cluster_idx, cluster_id in enumerate(cluster_order):
    print(f"\n处理簇 {cluster_id} ({cluster_idx+1}/{len(cluster_order)})...")
    
    # 获取簇内节点（排除仓库节点0）
    cluster_indices = np.where(cluster_labels == cluster_id)[0]
    cluster_nodes = data['node_ids'][cluster_indices]
    
    # 移除仓库节点（如果存在）
    if data['depot_id'] in cluster_nodes:
        depot_mask = cluster_nodes != data['depot_id']
        cluster_nodes = cluster_nodes[depot_mask]
        cluster_indices = cluster_indices[depot_mask]
    
    print(f"  簇内节点数: {len(cluster_nodes)}")
    
    if len(cluster_nodes) == 0:
        print("  空簇，跳过...")
        intra_cluster_paths.append([])
        continue
    
    # 准备簇内数据
    cluster_data = {
        'node_ids': data['node_ids'][cluster_indices],
        'a_i': data['a_i'][cluster_indices],
        'b_i': data['b_i'][cluster_indices],
        's_i': data['s_i'][cluster_indices],
        'travel_time_matrix': data['travel_time_matrix'][cluster_indices][:, cluster_indices],
        'num_nodes': len(cluster_indices)
    }
    
    # 调用滑动窗口优化
    print("  执行滑动窗口优化...")
    window_results = sliding_window_optimization(
        data=cluster_data,
        window_size=window_size,
        step_size=step_size,
        lambda_time=1.0,
        lambda_distance=1.0,
        max_iterations=50
    )
    
    # 提取结果
    cluster_path = window_results['optimized_path']
    cluster_distance = window_results['total_distance']
    cluster_penalty = window_results['total_penalty']
    
    # 存储结果
    intra_cluster_paths.append(cluster_path)
    total_intra_distance += cluster_distance
    total_penalty += cluster_penalty
    
    print(f"  簇内路径长度: {len(cluster_path)} 个节点")
    print(f"  簇内距离: {cluster_distance:.2f}")
    print(f"  簇内惩罚: {cluster_penalty:.2f}")

print("\n" + "="*50)
print("簇内滑动窗口优化完成！")
print(f"总簇内距离: {total_intra_distance:.2f}")
print(f"总惩罚值: {total_penalty:.2f}")
print(f"综合成本: {total_intra_distance + total_penalty:.2f}")

In [ ]:
## 4. 迭代改进循环

# 步骤5: 迭代改进循环
print("\n" + "="*50)
print("开始迭代改进循环...")

# 准备迭代改进所需的数据结构
current_solution = {
    'cluster_labels': cluster_labels.copy(),
    'cluster_order': cluster_order.copy(),
    'intra_cluster_paths': intra_cluster_paths.copy(),
    'total_distance': total_intra_distance + inter_cluster_distance,
    'total_penalty': total_penalty
}

print(f"初始解综合成本: {current_solution['total_distance'] + current_solution['total_penalty']:.2f}")

# 调用迭代改进控制器
print("\n执行迭代改进...")
refinement_results = iterative_refinement_controller(
    data=data,
    initial_solution=current_solution,
    max_iterations=5,
    convergence_threshold=0.01,
    window_size=window_size,
    step_size=step_size
)

# 提取改进后的结果
improved_solution = refinement_results['final_solution']
iteration_history = refinement_results['iteration_history']
converged = refinement_results['converged']

print(f"\n迭代改进完成！收敛状态: {converged}")
print(f"最终综合成本: {improved_solution['total_distance'] + improved_solution['total_penalty']:.2f}")
print(f"改进幅度: {((current_solution['total_distance'] + current_solution['total_penalty']) - (improved_solution['total_distance'] + improved_solution['total_penalty'])):.2f}")

# 更新变量
cluster_labels = improved_solution['cluster_labels']
cluster_order = improved_solution['cluster_order']
intra_cluster_paths = improved_solution['intra_cluster_paths']
total_intra_distance = improved_solution['total_distance'] - inter_cluster_distance
total_penalty = improved_solution['total_penalty']

# 可视化迭代过程
print("\n生成迭代过程可视化...")
fig = plot_iteration_progress(iteration_history)
plt.tight_layout()
plt.show()

print("迭代改进循环完成！")

In [ ]:
## 5. 路径拼接与后处理

# 步骤6: 路径拼接与后处理
print("\n" + "="*50)
print("开始路径拼接与后处理...")

# 构建完整路径：0 → 簇1路径 → ... → 簇K路径 → 0
print("构建完整路径...")
full_path = [data['depot_id']]  # 从仓库开始

# 按簇间顺序拼接簇内路径
for cluster_id in cluster_order:
    cluster_path = intra_cluster_paths[cluster_id]
    if cluster_path:  # 非空路径
        full_path.extend(cluster_path)

# 返回仓库
full_path.append(data['depot_id'])

print(f"完整路径长度: {len(full_path)} 个节点（包含起点和终点的仓库）")
print(f"完整路径: {full_path[:10]}...{full_path[-10:] if len(full_path) > 20 else ''}")

# 计算详细时间窗违反情况
print("\n计算时间窗违反情况...")
arrival_times = [0]  # 从时间0开始
violations = []
early_count = 0
late_count = 0
on_time_count = 0

current_time = 0
for i in range(1, len(full_path) - 1):  # 跳过起点和终点的仓库
    node_id = full_path[i]
    node_idx = np.where(data['node_ids'] == node_id)[0][0]
    
    # 旅行时间
    prev_node_id = full_path[i-1]
    prev_node_idx = np.where(data['node_ids'] == prev_node_id)[0][0] if prev_node_id != data['depot_id'] else data['depot_id']
    travel_time = data['travel_time_matrix'][prev_node_idx, node_idx] if prev_node_id != data['depot_id'] else 0
    
    # 到达时间
    arrival_time = current_time + travel_time
    arrival_times.append(arrival_time)
    
    # 服务时间
    service_time = data['s_i'][node_idx]
    
    # 检查时间窗违反
    a_i = data['a_i'][node_idx]  # 最早时间
    b_i = data['b_i'][node_idx]  # 最晚时间
    
    if arrival_time < a_i:
        # 早到，需要等待
        violation = a_i - arrival_time
        violations.append({
            'node_id': node_id,
            'arrival_time': arrival_time,
            'time_window': [a_i, b_i],
            'violation_type': 'early',
            'violation_amount': violation
        })
        early_count += 1
        # 等待到最早时间
        current_time = a_i + service_time
    elif arrival_time > b_i:
        # 晚到，违反时间窗
        violation = arrival_time - b_i
        violations.append({
            'node_id': node_id,
            'arrival_time': arrival_time,
            'time_window': [a_i, b_i],
            'violation_type': 'late',
            'violation_amount': violation
        })
        late_count += 1
        current_time = arrival_time + service_time
    else:
        # 准时到达
        on_time_count += 1
        current_time = arrival_time + service_time

# 返回仓库的旅行时间
last_node_id = full_path[-2]
last_node_idx = np.where(data['node_ids'] == last_node_id)[0][0]
return_travel_time = data['travel_time_matrix'][last_node_idx, data['depot_id']]
arrival_times.append(current_time + return_travel_time)

print("\n时间窗违反统计:")
print(f"  准时到达: {on_time_count} 个节点 ({on_time_count/(len(full_path)-2)*100:.1f}%)")
print(f"  提前到达: {early_count} 个节点 ({early_count/(len(full_path)-2)*100:.1f}%)")
print(f"  延迟到达: {late_count} 个节点 ({late_count/(len(full_path)-2)*100:.1f}%)")

if violations:
    total_violation = sum(v['violation_amount'] for v in violations)
    avg_violation = total_violation / len(violations) if violations else 0
    print(f"  总违反量: {total_violation:.2f}")
    print(f"  平均违反量: {avg_violation:.2f}")
    
    # 显示最严重的违反
    print("\n最严重的时间窗违反（前5个）:")
    sorted_violations = sorted(violations, key=lambda x: x['violation_amount'], reverse=True)[:5]
    for i, v in enumerate(sorted_violations):
        print(f"  {i+1}. 节点 {v['node_id']}: {v['violation_type']} {v['violation_amount']:.2f} (到达时间 {v['arrival_time']:.2f}, 时间窗 [{v['time_window'][0]:.1f}, {v['time_window'][1]:.1f}])")
else:
    print("  无时间窗违反！")

print("\n路径拼接与后处理完成！")

In [ ]:
## 6. 结果可视化与文件输出

In [ ]:
# 步骤7: 结果可视化
print("\n" + "="*50)
print("生成结果可视化...")

# 1. 层次化路径可视化
print("1. 生成层次化路径可视化...")
fig1 = plot_hierarchical_path(
    mds_coords=mds_coords,
    cluster_labels=cluster_labels,
    cluster_order=cluster_order,
    intra_cluster_paths=intra_cluster_paths,
    node_ids=data['node_ids'],
    full_path=full_path
)
plt.tight_layout()
plt.show()

# 2. 窗口优化过程可视化（第一个簇）
print("\n2. 生成窗口优化过程可视化（第一个簇）...")
if intra_cluster_paths and len(intra_cluster_paths[0]) > 0:
    # 获取第一个簇的数据
    first_cluster_id = cluster_order[0]
    cluster_indices = np.where(cluster_labels == first_cluster_id)[0]
    cluster_data = {
        'node_ids': data['node_ids'][cluster_indices],
        'a_i': data['a_i'][cluster_indices],
        'b_i': data['b_i'][cluster_indices],
        'travel_time_matrix': data['travel_time_matrix'][cluster_indices][:, cluster_indices]
    }
    
    fig2 = plot_window_optimization(
        data=cluster_data,
        window_size=window_size,
        step_size=step_size,
        optimized_path=intra_cluster_paths[first_cluster_id]
    )
    plt.tight_layout()
    plt.show()
else:
    print("  第一个簇为空，跳过窗口优化可视化")

# 3. 最终路径拓扑可视化（使用问题2的plot_final_route_pro或类似函数）
print("\n3. 生成最终路径拓扑可视化...")
try:
    # 尝试导入问题2的可视化函数
    from problem2_utils import plot_final_route_pro
    
    # 准备数据
    route_data = {
        'path': full_path,
        'node_ids': data['node_ids'],
        'arrival_times': arrival_times,
        'time_windows': list(zip(data['a_i'], data['b_i']))
    }
    
    fig3 = plot_final_route_pro(route_data)
    plt.tight_layout()
    plt.show()
    
except ImportError:
    print("  未找到plot_final_route_pro函数，使用替代可视化")
    # 创建简单的时间线图
    fig3, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))
    
    # 时间线
    ax1.plot(range(len(arrival_times)), arrival_times, 'b-o', linewidth=2, markersize=4)
    ax1.set_xlabel('节点顺序')
    ax1.set_ylabel('到达时间')
    ax1.set_title('路径时间线')
    ax1.grid(True, alpha=0.3)
    
    # 时间窗遵守情况
    node_indices = range(1, len(full_path) - 1)  # 跳过仓库
    arrival_times_customer = arrival_times[1:-1]  # 跳过仓库
    
    for i, idx in enumerate(node_indices):
        node_id = full_path[idx]
        node_idx_in_data = np.where(data['node_ids'] == node_id)[0][0]
        a_i = data['a_i'][node_idx_in_data]
        b_i = data['b_i'][node_idx_in_data]
        
        # 绘制时间窗
        ax2.plot([i, i], [a_i, b_i], 'k-', linewidth=2)
        # 绘制到达时间
        if arrival_times_customer[i] < a_i:
            ax2.plot(i, arrival_times_customer[i], 'go', markersize=6)  # 早到，绿色
        elif arrival_times_customer[i] > b_i:
            ax2.plot(i, arrival_times_customer[i], 'ro', markersize=6)  # 晚到，红色
        else:
            ax2.plot(i, arrival_times_customer[i], 'bo', markersize=6)  # 准时，蓝色
    
    ax2.set_xlabel('节点索引')
    ax2.set_ylabel('时间')
    ax2.set_title('时间窗遵守情况（黑线：时间窗，蓝点：准时，绿点：早到，红点：晚到）')
    ax2.grid(True, alpha=0.3)
    ax2.legend(['时间窗', '准时', '早到', '晚到'], loc='upper right')
    
    plt.tight_layout()
    plt.show()

print("\n结果可视化完成！")

In [ ]:
# 步骤8: 文件输出
print("\n" + "="*50)
print("保存结果文件...")

import os

# 确保输出目录存在
output_dir = "output"
os.makedirs(output_dir, exist_ok=True)

# 1. 保存完整路径
print("1. 保存完整路径到 final_route_50nodes.csv...")
path_df = pd.DataFrame({
    'node_order': range(len(full_path)),
    'node_id': full_path,
    'arrival_time': arrival_times
})
path_file = os.path.join(output_dir, "final_route_50nodes.csv")
path_df.to_csv(path_file, index=False, encoding='utf-8-sig')
print(f"  已保存到: {path_file}")
print(f"  路径长度: {len(full_path)} 个节点")

# 2. 保存时间窗违反情况
print("\n2. 保存时间窗违反情况到 time_window_violations.csv...")
if violations:
    violations_df = pd.DataFrame(violations)
    violations_file = os.path.join(output_dir, "time_window_violations.csv")
    violations_df.to_csv(violations_file, index=False, encoding='utf-8-sig')
    print(f"  已保存到: {violations_file}")
    print(f"  违反数量: {len(violations)} 个节点")
else:
    print("  无时间窗违反，跳过保存")

# 3. 保存聚类结果
print("\n3. 保存聚类结果到 clustering_results.csv...")
clustering_df = pd.DataFrame({
    'node_id': data['node_ids'],
    'cluster_id': cluster_labels,
    'a_i': data['a_i'],
    'b_i': data['b_i'],
    's_i': data['s_i']
})
clustering_file = os.path.join(output_dir, "clustering_results.csv")
clustering_df.to_csv(clustering_file, index=False, encoding='utf-8-sig')
print(f"  已保存到: {clustering_file}")
print(f"  聚类数量: {num_clusters} 个簇")

# 4. 输出算法统计信息
print("\n4. 算法统计信息:")
total_distance = total_intra_distance + inter_cluster_distance
total_cost = total_distance + total_penalty

stats = {
    "总节点数": data['num_nodes'],
    "簇数量": num_clusters,
    "窗口大小": window_size,
    "步长": step_size,
    "迭代次数": len(iteration_history) if 'iteration_history' in locals() else 0,
    "簇间距离": inter_cluster_distance,
    "簇内总距离": total_intra_distance,
    "总旅行距离": total_distance,
    "总惩罚值": total_penalty,
    "综合成本": total_cost,
    "准时节点数": on_time_count,
    "提前节点数": early_count,
    "延迟节点数": late_count,
    "时间窗遵守率": f"{on_time_count/(len(full_path)-2)*100:.1f}%",
    "最大QUBO规模": f"{window_size**2} 量子比特",
    "硬件限制": "550 量子比特",
    "是否满足硬件限制": window_size**2 <= 550
}

for key, value in stats.items():
    print(f"  {key}: {value}")

# 5. 保存统计信息到文件
print("\n5. 保存算法统计信息到 algorithm_statistics.txt...")
stats_file = os.path.join(output_dir, "algorithm_statistics.txt")
with open(stats_file, 'w', encoding='utf-8') as f:
    f.write("大规模单车辆调度算法统计信息\n")
    f.write("="*50 + "\n\n")
    for key, value in stats.items():
        f.write(f"{key}: {value}\n")
    
    f.write("\n" + "="*50 + "\n")
    f.write("簇间访问顺序:\n")
    f.write(f"{cluster_order}\n\n")
    
    f.write("完整路径（前20个节点）:\n")
    f.write(f"{full_path[:20]}\n")
    if len(full_path) > 20:
        f.write(f"... 共 {len(full_path)} 个节点\n")

print(f"  已保存到: {stats_file}")
print("\n文件输出完成！")

## 7. 性能评估与总结

In [ ]:
# 步骤9: 性能评估与总结
print("\n" + "="*50)
print("性能评估与总结")

print("\n算法执行完成！以下是主要成果总结：")

# 计算关键性能指标
total_nodes_visited = len(full_path) - 2  # 排除起点和终点的仓库
time_window_compliance_rate = on_time_count / total_nodes_visited * 100 if total_nodes_visited > 0 else 0
distance_efficiency = total_distance / total_nodes_visited if total_nodes_visited > 0 else 0
cost_per_node = total_cost / total_nodes_visited if total_nodes_visited > 0 else 0

print(f"\n1. 路径覆盖:")
print(f"   • 总访问节点: {total_nodes_visited} / {data['num_nodes']} ({total_nodes_visited/data['num_nodes']*100:.1f}%)")
print(f"   • 完整路径长度: {len(full_path)} 个节点（包含仓库）")

print(f"\n2. 时间窗性能:")
print(f"   • 准时到达率: {time_window_compliance_rate:.1f}%")
print(f"   • 提前到达: {early_count} 个节点 ({early_count/total_nodes_visited*100:.1f}%)")
print(f"   • 延迟到达: {late_count} 个节点 ({late_count/total_nodes_visited*100:.1f}%)")
if violations:
    print(f"   • 总时间窗违反量: {sum(v['violation_amount'] for v in violations):.2f}")

print(f"\n3. 距离效率:")
print(f"   • 总旅行距离: {total_distance:.2f}")
print(f"   • 平均每节点距离: {distance_efficiency:.2f}")
print(f"   • 簇间距离占比: {inter_cluster_distance/total_distance*100:.1f}%")
print(f"   • 簇内距离占比: {total_intra_distance/total_distance*100:.1f}%")

print(f"\n4. 成本分析:")
print(f"   • 总综合成本: {total_cost:.2f} (距离: {total_distance:.2f} + 惩罚: {total_penalty:.2f})")
print(f"   • 平均每节点成本: {cost_per_node:.2f}")

print(f"\n5. 聚类效果:")
print(f"   • 聚类数量: {num_clusters} 个簇")
print(f"   • 平均簇大小: {total_nodes_visited/num_clusters:.1f} 节点/簇")
print(f"   • 簇大小分布: ", end="")
for cluster_id in range(num_clusters):
    cluster_size = np.sum(cluster_labels == cluster_id)
    print(f"簇{cluster_id}:{cluster_size} ", end="")
print()

print(f"\n6. 硬件限制满足情况:")
print(f"   • 滑动窗口大小: {window_size} 个节点")
print(f"   • 最大QUBO规模: {window_size**2} 量子比特")
print(f"   • 硬件限制: 550 量子比特")
print(f"   • 是否满足限制: {'是' if window_size**2 <= 550 else '否'}")

print(f"\n7. 迭代改进效果:")
if 'iteration_history' in locals() and iteration_history:
    initial_cost = iteration_history[0]['total_cost']
    final_cost = iteration_history[-1]['total_cost']
    improvement = (initial_cost - final_cost) / initial_cost * 100
    print(f"   • 初始成本: {initial_cost:.2f}")
    print(f"   • 最终成本: {final_cost:.2f}")
    print(f"   • 改进幅度: {improvement:.2f}%")
    print(f"   • 迭代次数: {len(iteration_history)}")
    print(f"   • 收敛状态: {'已收敛' if converged else '未收敛'}")
else:
    print("   • 迭代改进未执行或数据不可用")

print(f"\n8. 输出文件:")
print(f"   • final_route_50nodes.csv - 完整路径数据")
print(f"   • time_window_violations.csv - 时间窗违反数据")
print(f"   • clustering_results.csv - 聚类结果数据")
print(f"   • algorithm_statistics.txt - 算法统计信息")

print("\n" + "="*50)
print("算法执行成功完成！")
print("所有模块已集成，完整工作流程已验证。")
print("结果文件已保存到 output/ 目录。")